### Coleta de dados de Temperatura, Umidade e Precipitação

Será utilizado o dataset derived-era5-single-levels-daily-statistics do ERA5

Documentação em:
https://cds.climate.copernicus.eu/datasets/derived-era5-single-levels-daily-statistics?tab=documentation

Os dados extraídos:
<pre>
- Umidade   -> variável 2m_dewpoint_temperature (ponto de orvalho),
               O processo de conversão para percentual será detalhado no passo que executa a conversão
</pre>

Os dados serão coletados por Ano e Mês 

Os dados requisitados estão no retangulo geográfico geográfico [6, -74, -34, -35] -> [Norte, Oeste, Sul, Leste] em graus onde está o Brasil


In [ ]:
import cdsapi
import os, sys
import xarray as xr
from datetime import datetime, timedelta
from pyspark.sql import functions as F

In [ ]:
# Cria a conexão Spark

# Adiciona a pasta raiz do projeto (um ou dois níveis acima) no caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))  # Ajuste a quantidade de '..' conforme a profundidade da subpasta

# Cria uma conexão Spark 
from spark_utils import get_spark_session # ver em C:\Marco Conti\Projetos\MAIS-v2\spark_utils.py
spark = get_spark_session("MeuNotebook")

In [ ]:
# Os arquivos utilizados durante o processamento serão removidos no final do notebook
remover_arquivos = []

Requisição dos dados da variável derived-era5-single-levels-daily-statistics do ERA5 utilizando a biblioteca cdsapi

In [ ]:
PROJECT_PATH   = os.getcwd()
DATA_PATH_ROOT = "C:\\Marco Conti\\Projetos\\Dados\\"

print(PROJECT_PATH)
print(DATA_PATH_ROOT)

In [ ]:
def obter_mes_dia(ano: int):
    hoje = datetime.now()

    # Se dia 01 ou 02, considerar mês anterior
    if hoje.day in (1, 2):
        data_referencia = hoje.replace(day=1) - timedelta(days=1)
    else:
        data_referencia = hoje

    ano_ref = data_referencia.year
    mes_ref = data_referencia.month

    # Lista de meses
    if ano < ano_ref:
        meses = [f"{m:02d}" for m in range(1, 13)]
        dias = [f"{d:02d}" for d in range(1, 32)]

    elif ano == ano_ref:
        meses = [f"{m:02d}" for m in range(1, mes_ref + 1)]

        # Último dia válido considerando D-2
        data_limite = hoje - timedelta(days=2)

        # Se estiver no mês da data limite, retorna apenas até D-2
        dias = [f"{d:02d}" for d in range(1, data_limite.day + 1)]

    else:
        raise ValueError(f"Ano futuro não permitido: {ano}")

    return meses, dias

def get_EAC4(year, client):

    month_list, day_list = obter_mes_dia(year)

    dataset = "reanalysis-era5-single-levels"
    request = {
        "product_type": "reanalysis",
        "variable": [
            "2m_dewpoint_temperature"
        ],
        "year":  f"{year}",
        "month": month_list,
        "day":   day_list,
        "time": ["10:00"],
        "data_format": "netcdf",
        "download_format": "unarchived",

        # "daily_statistic": "daily_mean",
        # "time_zone": "utc-03:00",
        # "frequency": "1_hourly",

        # Retangulo geográfico definido por Norte, Oeste, Sul e Leste em graus onde está o Brasil
        "area": [6      # Norte
                ,-74    # Oeste
                ,-34    # Sul
                ,-38]   # Leste
    }

    ret_download = client.retrieve(dataset, request).download()
    return ret_download

def convert_netcdf4_Spark(file_name):
    with xr.open_dataset(file_name
                        ,engine="netcdf4"
                        # ,chunks={"time": 365
                        #         ,"latitude": 100
                        #         ,"longitude": 100 }
                        ) as ds:
        
        # Transforma o Dataset em um Spark Dataframe
        df_dask   = ds.to_dask_dataframe()
        df_dask_c = df_dask.compute()
        df_spark  = spark.createDataFrame(df_dask_c)    
    return df_spark    

def convert_unit(df_ponto_orvalho):
    drop_cols = ["valid_time", "number", "d2m"]
    df_ponto_orvalho_celsius = \
        (df_ponto_orvalho
            .withColumns({"indicador": F.lit("ponto_orvalho")
                        ,"valor": (F.col("d2m") - F.lit(273.15)).cast('double')
                        ,"unidade_medida": F.lit("celsius")
                        ,"data_medicao": F.col("valid_time").cast("date")}
                        )
            .drop(*drop_cols)
        )
    return df_ponto_orvalho_celsius

def write_data_csv(df_ponto_orvalho_celsius, write_path, file_name):
    df_ponto_orvalho_celsius.toPandas().to_csv(f"{write_path}\{file_name}")

In [8]:
client = cdsapi.Client(
    url = os.getenv("ECMWF_DATASTORES_URL"),
    key = os.getenv("ECMWF_DATASTORES_KEY"),
)

for year in range(2026, 2027): # Ajuste o intervalo de anos conforme necessário
    # for month in range(1,13):
    start = datetime.now()
    print("\nStart download - year: ", year, start)    

    ret_download = get_EAC4(year, client)
    nc_file_name = r"{DATA_PATH_ROOT}ERA5-Umidade\arquivos_nc\ERA5_umidade_{year}.nc".format(DATA_PATH_ROOT = DATA_PATH_ROOT, year = year)
    os.rename(ret_download, nc_file_name)

    df_ponto_orvalho = convert_netcdf4_Spark(nc_file_name)

    df_ponto_orvalho_celsius = convert_unit(df_ponto_orvalho)

    csv_path      = r"{DATA_PATH_ROOT}\ERA5-Umidade\arquivos_csv".format(DATA_PATH_ROOT = DATA_PATH_ROOT)
    csv_file_name = f"ERA5_ponto_orvalho_{year}.csv"

    write_data_csv(df_ponto_orvalho_celsius, csv_path, csv_file_name)

    final = datetime.now()
    print("End download and transformations : ", final, " - Duration: ", final-start, "\n")

    # time.sleep(300)


Start download - year:  2026 2026-08-07 15:03:49.779517


2026-08-07 15:03:50,469 INFO Request ID is 4e7c107d-1d7a-46b0-81ae-8792463dc35e
2026-08-07 15:03:51,155 INFO status has been updated to accepted
2026-08-07 15:04:14,621 INFO status has been updated to running
2026-08-07 15:04:26,181 INFO status has been updated to successful


End download and transformations :  2026-08-07 15:04:46.520130  - Duration:  0:00:56.740613 



Esta função ira converter os dados dos arquivos .nc para o format Dask para então converter para Dataframe Spark <br>
Isso deixa o processamento em paralelo e será importante para processamento de grandes volumes (1 ano com todos os meses e dias)

Faz a junção dos dados de temperatra e ponto de orvalho para calcular o percentual da umidade:

In [ ]:
df_temperatura = spark.read.parquet(r"C:\Marco Conti\Projetos\MAIS-v2\dados\ERA5-temperaturas\ERA5_temperatura.parquet")
print("Temperatura: ", df_temperatura.count())          # 783.587
print("Ponto de orvalho:", df_ponto_orvalho.count())    # 379.155

In [ ]:
df_temp_ponto_orvalho = \
    (df_temperatura.alias('t')
        .join(df_ponto_orvalho.alias('p')
             ,((F.col('t.data_medicao') == F.col('p.data_medicao')) & 
               (F.col("t.latitude")     == F.col("p.latitude")) & 
               (F.col("t.longitude")    == F.col("p.longitude")))
             ,'inner')
        .select('t.data_medicao'
               ,'t.latitude'
               ,'t.longitude'
               ,F.col('t.valor').alias('temperatura_celsius')
               ,F.col('p.valor').alias('temp_ponto_orvalho_celsius'))
    )


print("Join:", df_temp_ponto_orvalho.count())

In [ ]:
df_temp_ponto_orvalho.limit(10).show(truncate=False)

Faz a conversão de temperatura para percentual de umidade usando a equação de Magnus-Tetens.

https://en.wikipedia.org/wiki/Tetens_equation


In [ ]:

# Constantes da equação de Magnus-Tetens.
A = 17.67
B = 243.5

df_umidade_Magnus_Tetens = (
    df_temp_ponto_orvalho
        .withColumn("indicador", F.lit("umidade"))
        .withColumn("unidade_medida", F.lit("percentual"))
        .withColumn("valor",
            F.round(  
                F.lit(100.0) * F.exp(
                    (
                        A * F.col("temp_ponto_orvalho_celsius") /
                        (F.col("temp_ponto_orvalho_celsius") + B)
                    )
                    -
                    (
                        A * F.col("temperatura_celsius") /
                        (F.col("temperatura_celsius") + B)
                    )
                )
            ,4)
        )
).drop("temperatura_celsius", "temp_ponto_orvalho_celsius")

df_umidade_Magnus_Tetens.show()

In [ ]:
df_umidade = \
    (df_umidade_Magnus_Tetens
        .select("data_medicao"
               ,"latitude"
               ,"longitude"
               ,"indicador"
               ,"valor"
               ,"unidade_medida"))

In [ ]:
# df_umidade.toPandas().to_csv("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\ERA5-temperaturas\\ERA5_umidade.csv", index=False)

df_umidade.toPandas().to_parquet("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\ERA5-temperaturas\\ERA5_umidade.parquet")

In [ ]:
# df_csv = spark.read.csv("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\ERA5-temperaturas\\ERA5_umidade_temperatura_precipitacao.csv", header=True, inferSchema=True)
# print("df_csv:", df_csv.count())
# df_csv.printSchema()
# df_csv.show(10,False)


df_parquet = spark.read.parquet("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\ERA5-temperaturas\\ERA5_umidade.parquet")
print("df_csv:", df_parquet.count())
df_parquet.printSchema()
df_parquet.show(10,False)


In [ ]:
# **** INCLUIR EXCLUSÃO DE ARQUIVOS (.zip e .nc)

for file in remover_arquivos:
    print("Arquivo:", file, end="")
    os.remove(file)
    print(" Removido com sucesso")
